# 11 (DS) — Exploratory Data Analysis

**Data Scientist perspective.** The first hour with a new dataset: profile it, map missing data, measure distributions and relationships, and hand off to pandas for plotting. Every step pushes down to IRIS — only aggregated results cross the wire.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. A realistic dataset

500 orders across regions and categories, with planted NULLs and outliers so profiling has something to find.

In [ ]:
import random
import pandas as pd

random.seed(42)
REGIOES = ["SP", "RJ", "MG", "RS"]
CATEGORIAS = ["eletronico", "moveis", "vestuario", "alimentos"]

rows = []
for i in range(1, 501):
    valor = round(random.lognormvariate(4.5, 0.6), 2)
    if i % 97 == 0:            # planted outliers
        valor = valor * 12
    rows.append({
        "pedido_id": i,
        "cliente_id": random.randint(1, 120),
        "regiao": random.choice(REGIOES),
        "categoria": random.choice(CATEGORIAS),
        "valor": None if i % 23 == 0 else valor,
        "desconto": None if i % 41 == 0 else round(random.random() * 0.3, 2),
    })

pedidos = session.createDataFrame(pd.DataFrame(rows))
print("rows:", pedidos.count())
pedidos.show(5)

## 2. Structure & summary statistics

`describe`/`summary` give per-column counts, means, and quantiles without leaving SQL.

In [ ]:
pedidos.printSchema()
pedidos.select("valor", "desconto").describe().show()
pedidos.summary().show()

## 3. Missing-data map

One aggregate per column — computed inside IRIS, returned as a single row.

In [ ]:
from irispark.functions import col, count, when

cols = pedidos.columns
null_profile = pedidos.agg(
    *[count(when(col(c).isNull(), c)).alias(f"{c}_nulls") for c in cols]
)
null_profile.show()

## 4. Distribution shape

IRISPARK-native aggregates run as UDAFs / analytic queries inside IRIS: median, percentiles, skewness, kurtosis.

In [ ]:
from irispark.functions import median, percentile, skewness, kurtosis, min, max

pedidos.select(
    min("valor").alias("min"),
    percentile("valor", 0.25).alias("p25"),
    median("valor").alias("median"),
    percentile("valor", 0.75).alias("p75"),
    percentile("valor", 0.90).alias("p90"),
    max("valor").alias("max"),
    skewness("valor").alias("skew"),
    kurtosis("valor").alias("kurt"),
).show()

## 5. Outlier peek above P90

Collect one scalar, then filter with it — two round-trips, zero data movement.

In [ ]:
p90 = pedidos.select(percentile("valor", 0.90).alias("v")).first()["v"]
outliers = pedidos.filter(col("valor") > p90)
print("P90:", p90, "| rows above:", outliers.count())
outliers.orderBy("valor DESC").select("pedido_id", "categoria", "valor").show(5)

## 6. Relationships: correlation & covariance

In [ ]:
print("corr(valor, desconto):", pedidos.stat.corr("valor", "desconto"))
print("cov(valor, desconto):", pedidos.stat.cov("valor", "desconto"))

## 7. Category crossings

`crosstab` for co-occurrence (returns a pandas DataFrame), `freqItems` for dominant values.

In [ ]:
xtab = pedidos.stat.crosstab("regiao", "categoria")
print(type(xtab).__name__)
print(xtab)
print("freqItems:", pedidos.stat.freqItems())

## 8. Handoff to pandas for plotting

Aggregate in IRIS, plot locally — only the small result crosses the wire.

In [ ]:
by_categoria = pedidos.groupBy("categoria").count().orderBy("categoria").to_pandas()
print(by_categoria)

try:
    import matplotlib.pyplot as plt
    by_categoria.plot.bar(x="categoria", y="count", legend=False,
                          title="Pedidos por categoria")
    plt.tight_layout()
    plt.savefig("eda_by_categoria.png")
    print("saved eda_by_categoria.png")
except ImportError:
    print("matplotlib not installed; skipping chart")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")